In [4]:
import os
import pandas as pd
from dbfread import DBF
import io

# 모든 폴더, 파일 리스트 추출
def list_folder_recursive(path):
    """
    재귀적으로 경로 내의 모든 파일과 폴더(절대경로)를 리스트로 반환
    반환 항목: (root, dirs, files) 구조를 원하면 다른 형태로 변경 가능
    """
    results = []
    for root, dirs, files in os.walk(path):
        for d in dirs:
            results.append(os.path.join(root, d))
        for f in files:
            results.append(os.path.join(root, f))
    return results

shp_folder_path = r'Z:\105. 정밀도로지도 검사(2026)\02. 계획노선도검토\3지구\2차\##오류내역\260406'

err_dbf_list = []

all_items = list_folder_recursive(shp_folder_path)
# print(len(all_items), "items found")
all_items_df = pd.DataFrame(all_items, columns=['path'])
# all_items_df.to_csv('all_items_list.csv', index=False, encoding='utf-8-sig')

# na=False 로 NaN 처리, r'\\.dbf' 또는 r'\.dbf'로 점(.)을 이스케이프하여 정확히 '.dbf' 매칭
cond1 = (
    # all_items_df['path'].str.contains('속성분류일치검사_부') &
    all_items_df['path'].str.contains('SEC') &
    all_items_df['path'].str.contains('.dbf')
)

all_items_df_filtered = all_items_df[cond1]
# all_items_df_filtered.to_csv('filtered_items_list.csv', index=False, encoding='utf-8-sig')
for i in range(len(all_items_df_filtered)):
    err_dbf_list.append(all_items_df_filtered.iloc[i, 0])

# 사업자 shp 전체 concat

# 여러개의 파일을 한번에 읽어 각각의 변수에 저장 - 딕셔너리로 진행
err_dbf_dict = {}
for i, path in enumerate(err_dbf_list):
    table = DBF(err_dbf_list[i], encoding='utf-8')
    # err_dbf_dict[f'df_dbf_{i}'] = pd.DataFrame(iter(table))
    err_dbf_dict[err_dbf_list[i].split('\\')[5] + '_' + err_dbf_list[i].split('\\')[6] + ' ' +f'{i}'] = pd.DataFrame(iter(table))


df_list = []

for key, value in err_dbf_dict.items():
    # temp_df = pd.concat(err_dbf_dict.values(), ignore_index=True)
    temp_df = value.copy()
    temp_df = temp_df[['error_expl']] # 필요한 정보만 추출
    
    # 키(지역명)를 새로운 컬럼으로 추가
    temp_df['region_id'] = key
    
    df_list.append(temp_df)

# df_dbf_all = pd.concat(err_dbf_dict.values(), ignore_index=True)
df_dbf_all = pd.concat(df_list, ignore_index=True)
print("사업자 shp Total : ", df_dbf_all.shape)

# df_dbf_all.to_csv('ERR_dbf.csv', index=False, encoding='utf-8-sig')

KeyError: "None of [Index(['error_expl'], dtype='object')] are in the [columns]"

In [ ]:
df_dbf_all[df_dbf_all['error_expl'].str.contains('')]['region_id'].unique()

array([], dtype=object)

In [2]:
df_dbf_all[df_dbf_all['error_expl'].str.contains('RM1')]['error_expl'].unique()

array(['RM1 링크매칭오류', 'RM1 링크매칭누락', 'RM1 링크 매칭 누락', 'RM1 TYPE오류',
       'RM1 단절 오류', 'RM1 단절오류', 'RM1 삭제 필요', 'RM1 KIND오류',
       'RM1 KIND오류, 링크매칭오류', 'RM1 단절 누락', 'RM1 TYPE, KIND 오류',
       'RM1 KIND오류, 단절 후 링크매칭 필요', 'RM1 KIND 오류', 'RM1링크매칭누락',
       'RM1 KIND, TYPE오루', '해당라인 RM1, SF1, RS1 병합 필요',
       'RM1 단절 후 선 하나로 수정', 'RM1 단절 필요', 'RM1 단절누락', 'RM1 링크 매칭 오류',
       'RM1 링크 매치 누락', 'RM1 단절 위치 오류', 'RM1  단절 오류', 'RM1 kind오류',
       'RM1 type 오류', 'RM1링크매칭오류', 'RM1단절오류', 'RM1 단절 후 하나로 수정, 링크매칭 수정',
       'RM1 단절 삭제'], dtype=object)

In [11]:
cond1 = df_dbf_all['error_expl'].str.contains('RM1')
cond2 = df_dbf_all['error_expl'].str.contains('KIND')
df_dbf_all[cond1 & cond2]

,error_expl,region_id
2370,RM1 KIND오류,지방도_514호선_SEC301 36
2371,"RM1 KIND오류, 링크매칭오류",지방도_514호선_SEC301 36
2376,"RM1 TYPE, KIND 오류",지방도_514호선_SEC301 36
2440,"RM1 KIND오류, 단절 후 링크매칭 필요",지방도_514호선_SEC302 38
2461,RM1 KIND오류,지방도_514호선_SEC302 38
2462,RM1 KIND 오류,지방도_514호선_SEC302 38
2498,"RM1 KIND, TYPE오루",지방도_514호선_SEC302 38
